In [ ]:
! git clone https://github.com/magewade/autoparts-recognition.git
%cd autoparts-recognition
! pip install -q -r requirements.txt
!playwright install
!apt-get update
!apt-get install -y libwoff2-1 libgstreamer-gl1.0-0 libgstreamer-plugins-base1.0-0 \
                     libavif13 libharfbuzz-icu0 libenchant-2-2 libsecret-1-0 \
                     libhyphen0 libmanette-0.2

Cloning into 'autoparts-recognition'...
remote: Enumerating objects: 652, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 652 (delta 21), reused 19 (delta 18), pack-reused 625 (from 1)
Receiving objects: 100% (652/652), 65.64 MiB | 13.03 MiB/s, done.
Resolving deltas: 100% (442/442), done.
Updating files: 100% (22/22), done.
/content/autoparts-recognition
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━

In [ ]:
# ! git clone https://github.com/magewade/autoparts-recognition.git
# %cd autoparts-recognition
# ! pip install -q -r requirements.txt

In [ ]:
!python main_goofish.py \
--api-keys key_1 key_2 \
--save-file-name 'predicted_data_goofish' \
--gemini-api-model 'gemini-2.5-flash' \
--desc-model 'gemini-2.5-flash-lite' \
--one-many-model 'gemini-2.0-flash-lite' \
--car-brand 'all' \
--max-steps 5 \
--max-links 100

2025-09-10 11:37:41.625916: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757504261.669335    2575 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757504261.691102    2575 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757504261.739857    2575 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757504261.739902    2575 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757504261.739912    2575 computation_placer.cc:177] computation placer alr

In [ ]:
import pandas as pd

df_many_desc = pd.read_csv("/content/autoparts-recognition/products_many_desc.csv")
df_many_desc.shape

(14, 5)

In [ ]:
df_many_image = pd.read_csv("/content/autoparts-recognition/products_many_image.csv")
df_many_image.shape

(37, 6)

In [ ]:
df_many_final = pd.read_csv("/content/autoparts-recognition/products_many_final.csv")
df_many_final.shape

(0, 8)

In [ ]:
df_final = pd.read_csv("/content/autoparts-recognition/products_one_final.csv")
df_final.shape

(49, 8)

In [ ]:
def safe_split(val, sep="|", n=3):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return [None] * n

    if isinstance(val, str):
        # убираем служебные маркеры
        val = val.replace("<START>", "").replace("<END>", "").strip()

        # если нет квадратных скобок, значит обычный вариант
        if "[" not in val and "]" not in val:
            parts = [x.strip() for x in val.split(sep)]
            while len(parts) < n:
                parts.append(None)
            return parts[:n]

    return [val] + [None] * (n - 1)


df_final[["desc_model", "desc_number", "desc_one_many"]] = df_final["description_model_guess"].apply(safe_split).apply(pd.Series)
df_final[["image_model", "image_number", "image_one_many"]] = df_final["extracted_number_from_barcode_image"].apply(safe_split).apply(pd.Series)
df_final = df_final.drop(columns=["description_model_guess", "extracted_number_from_barcode_image", "description"])

# Сохраняем в Excel
df_final.to_excel("dataset_one.xlsx", index=False)
print("✅ Done!")

✅ Done!


In [ ]:
merged_df = pd.concat([df_many_desc, df_many_image, df_many_final],
                      ignore_index=True, sort=False)
merged_df = merged_df.where(pd.notnull(merged_df), None)

merged_df[["desc_model", "desc_number", "desc_one_many"]] = merged_df["description_model_guess"].apply(safe_split).apply(pd.Series)
merged_df[["image_model", "image_number", "image_one_many"]] = merged_df["extracted_number_from_barcode_image"].apply(safe_split).apply(pd.Series)
merged_df = merged_df.drop(columns=["description_model_guess", "extracted_number_from_barcode_image", "description"])
merged_df.to_excel("dataset_many.xlsx", index=False)

print("✅ Done!")

✅ Done!
